In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
import aotools
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator, FuncFormatter
import VisAnalysis as Vis
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.interpolate import interp1d
%matplotlib qt
import matplotlib.colors as mcolors
from matplotlib import transforms
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter

def frac_to_percent(x, pos):
    return f"{int(round(100 * x))}"

def _finite(a):
    a = np.asarray(a)
    return a[np.isfinite(a)]

def _rgba(color, alpha):
    return mcolors.to_rgba(color, alpha=alpha)

def choose_bins_freedman_diaconis(data, data_min, data_max,
                                  min_bins=80, max_bins=260):
    data = _finite(data)
    n = data.size
    if n < 2:
        return min_bins

    q25, q75 = np.percentile(data, [25, 75])
    iqr = q75 - q25
    data_range = float(data_max - data_min)

    if iqr <= 0 or data_range <= 0:
        nb = int(np.sqrt(n))
    else:
        bin_width = 2 * iqr * (n ** (-1 / 3))
        nb = int(np.ceil(data_range / bin_width)) if bin_width > 0 else int(np.sqrt(n))

    return int(np.clip(nb, min_bins, max_bins))


def fraction_hist_with_stats(
    axh, data, bins, base_color,
    outline_alpha=0.75,
    outline_lw=2.0,
):
    """
    Unshaded histogram (outline only), normalized to fraction per bin.
    Returns: (peak, mu, sigma)
    """
    data = _finite(data)
    if data.size == 0:
        return 0.0, np.nan, np.nan

    mu = float(np.mean(data))
    sigma = float(np.std(data))

    weights = np.ones_like(data, dtype=float) / data.size

    axh.hist(
        data,
        bins=bins,
        orientation="horizontal",
        weights=weights,
        histtype="step",
        color=_rgba(base_color, outline_alpha),
        linewidth=outline_lw,
        zorder=2
    )

    h, _ = np.histogram(data, bins=bins, weights=weights)
    peak = float(h.max()) if h.size else 0.0
    return peak, mu, sigma


def draw_mean_sigma_on_hist(axh, mu, sigma, color, cfg, series_index=0):
    """
    Draw mean as a short horizontal tick at y=mu near the RIGHT edge of histogram,
    and sigma as a vertical line (perpendicular) spanning [mu-sigma, mu+sigma].
    series_index lets you offset series (e.g. orange) so markers don't overlap.
    """
    if not np.isfinite(mu):
        return

    x0, x1 = axh.get_xlim()
    xmax = x1

    # Base x-position near RHS, then offset per series
    base_x = cfg["x_frac"] * xmax
    dx = cfg["dx_frac"] * xmax * series_index
    x = base_x + dx

    # Keep inside axis
    x = min(x, x1 - cfg["pad_frac"] * xmax)

    # Mean tick (horizontal)
    tick_half = 0.5 * cfg["mean_tick_frac"] * xmax
    axh.plot(
        [x - tick_half, x + tick_half],
        [mu, mu],
        color=color,
        alpha=cfg["mean_alpha"],
        linewidth=cfg["mean_lw"],
        solid_capstyle="butt",
        zorder=10,
        clip_on=False
    )

    # Sigma spread (vertical, perpendicular)
    if cfg.get("show_sigma", True) and np.isfinite(sigma) and sigma > 0:
        axh.plot(
            [x, x],
            [mu - sigma, mu + sigma],
            color=color,
            alpha=cfg["sigma_alpha"],
            linewidth=cfg["sigma_lw"],
            solid_capstyle="butt",
            zorder=9,
            clip_on=False
        )

def apply_legend(ax, handles, labels, legend_cfg):
    """
    legend_cfg examples:
      {"loc":"lower center", "bbox_to_anchor":(-1.45, 0.01), "fontsize":12, "frameon":True, "ncol":1}
      {"loc":"upper right"}
      {"enabled": False}
    """
    if legend_cfg is None:
        legend_cfg = {}
    if legend_cfg.get("enabled", True) is False:
        return None

    # defaults
    cfg = dict(fontsize=12, frameon=True)
    cfg.update(legend_cfg)

    # remove non-legend keys if present
    cfg.pop("enabled", None)

    return ax.legend(handles, labels, **cfg)

In [8]:
import numpy as np
from scipy.interpolate import interp1d
from sklearn.isotonic import IsotonicRegression


def delta_eta_from_voa_volts(
    voa_volts: np.ndarray,
    vs_cal: np.ndarray,
    psfixed_cal: np.ndarray,
    *,
    V0: float = None,
    assume_decreasing: bool = True,
) -> np.ndarray:
    """
    Convert VOA control voltage -> power imbalance Δη(t).

    Assumptions:
      - Constant arm power equals VOA arm power at V0 (default: mean voltage)
      - psfixed_cal measured during VOA sweep
      - Visibility imbalance defined as:
            Δη = (P_const - P_voa) / (P_const + P_voa)

    Parameters
    ----------
    voa_volts : array
        VOA voltage vs time
    vs_cal : array
        VOA voltages used in calibration sweep
    psfixed_cal : array
        Measured powers (or couplings) during calibration sweep
    V0 : float, optional
        Balance voltage; default = mean(voa_volts)
    assume_decreasing : bool
        True if power decreases with increasing voltage (typical VOA)

    Returns
    -------
    delta_eta : array
        Power imbalance Δη(t) in [-1, 1]
    """

    voa_volts = np.asarray(voa_volts, float)
    vs_cal = np.asarray(vs_cal, float)
    psfixed_cal = np.asarray(psfixed_cal, float)

    # Normalize power
    psnorm = psfixed_cal / np.max(psfixed_cal)

    # Enforce physical monotonicity via isotonic regression
    iso = IsotonicRegression(increasing=not assume_decreasing)
    ps_mono = iso.fit_transform(vs_cal, psnorm)

    # Voltage -> relative power interpolation
    voltage_to_p = interp1d(
        vs_cal, ps_mono,
        bounds_error=False,
        fill_value="extrapolate"
    )

    # Balance point
    if V0 is None:
        V0 = float(np.nanmean(voa_volts))

    P_t = voltage_to_p(voa_volts)
    P_0 = float(voltage_to_p(V0))

    # Relative power ratio
    r = P_t / P_0

    # Power imbalance Δη
    delta_eta = (1.0 - r) / (1.0 + r)

    return (r,delta_eta)


In [44]:
DNF = Vis.load_measurements(base_dir="D:\PhD Thesis Writing\Data", date_str="2025-11-26", subfolder="Day2LongNoFibre", verbose=True)

Loaded 12 measurements from D:\PhD Thesis Writing\Data\2025-11-26\Day2LongNoFibre


In [159]:
np.mean(DNF[8]['data']['voa_volts'])

2.6014077291752082

In [161]:
dnf_volts=DNF[0]['data']['voa_volts']

In [163]:
vs = np.load("vs.npy")
psfixed = np.load("psfixed.npy")

In [167]:
del_eta=delta_eta_from_voa_volts(
    voa_volts=dnf_volts,
    vs_cal= vs,
    psfixed_cal=psfixed,
    V0=2.60)

In [169]:
def delta_eta_to_loss_db(delta_eta, eps=1e-12):
    de = np.clip(delta_eta, -1 + eps, 1 - eps)
    return 10 * np.log10((1 + de) / (1 - de))

In [172]:
def delta_eta_to_loss_db2(r, eps=1e-12):
    return 10 * np.log10(r)

In [129]:
d1 = np.load('DNF_1000_em_Numba.npy', allow_pickle=True)
d2 = np.load('DNF_1000_voa_Numba.npy', allow_pickle=True)

In [141]:
dnf_volts

array([0., 0., 0., ..., 5., 5., 5.])

In [131]:
type(d2)

numpy.ndarray

In [133]:
d1[1]

VisibilityTauResult(taus=array([3.00e-04, 4.00e-04, 5.00e-04, 6.00e-04, 7.00e-04, 8.00e-04,
       9.00e-04, 1.00e-03, 2.00e-03, 3.00e-03, 4.00e-03, 5.00e-03,
       6.00e-03, 7.00e-03, 8.00e-03, 9.00e-03, 1.00e-02, 1.10e-02,
       1.20e-02, 1.30e-02, 1.40e-02, 1.50e-02, 1.60e-02, 1.70e-02,
       1.90e-02, 2.00e-02, 2.20e-02, 2.30e-02, 2.50e-02, 2.70e-02,
       2.80e-02, 3.10e-02, 3.30e-02, 3.50e-02, 3.80e-02, 4.00e-02,
       4.30e-02, 4.60e-02, 5.00e-02, 5.30e-02, 5.70e-02, 6.10e-02,
       6.60e-02, 7.10e-02, 7.60e-02, 8.10e-02, 8.70e-02, 9.30e-02,
       1.00e-01, 1.07e-01, 1.15e-01, 1.23e-01, 1.32e-01, 1.42e-01,
       1.52e-01, 1.63e-01, 1.75e-01, 1.87e-01, 2.01e-01, 2.15e-01,
       2.31e-01, 2.48e-01, 2.66e-01, 2.85e-01, 3.05e-01, 3.27e-01,
       3.51e-01, 3.76e-01, 4.04e-01, 4.33e-01, 4.64e-01, 4.98e-01,
       5.34e-01, 5.72e-01, 6.14e-01, 6.58e-01, 7.05e-01, 7.56e-01,
       8.11e-01, 8.70e-01, 9.33e-01, 1.00e+00]), n_windows=array([1999998,  500000,  400000,  333333,  2

In [135]:
d2

array([list([[array([1.500000e-03, 4.500000e-03, 7.500000e-03, ..., 1.999905e+02,
              1.999935e+02, 1.999965e+02]), array([0.85032321, 0.68571896, 0.74957374, ..., 0.27532131, 0.2677438 ,
              0.1138669 ]), {'dt': 0.0001, 'K_samples': 30, 'S_samples': 30, 'tau_eff_s': 0.003, 'step_eff_s': 0.003, 'n_windows_total': 66666, 'n_windows_used': 66666, 'n_phase': 64, 'iters_ab': 8, 'lam_floor': 1e-06, 'n_min_counts': 0.0, 'clip_for_qber': False}], [array([1.500000e-03, 4.350000e-02, 8.550000e-02, ..., 1.998795e+02,
              1.999215e+02, 1.999635e+02]), array([0.85032321, 0.55845347, 0.76547674, ..., 0.2384071 , 0.27545176,
              0.0712761 ])], [array([1.500000e-03, 4.350000e-02, 8.550000e-02, ..., 1.998795e+02,
              1.999215e+02, 1.999635e+02]), array([0.07483839, 0.22077326, 0.11726163, ..., 0.38079645, 0.36227412,
              0.46436195])]])                                                                                                            

In [192]:
DB_3 = delta_eta_to_loss_db2(del_eta[0])
ts_3 = [i / 10000 for i in range(len(DB_3))]

In [180]:
DDDD=DB_1/DB_3

In [178]:
plt.plot(ts_1,DB_1)

In [194]:
plt.plot(ts_3,DB_3)

In [182]:
plt.plot(ts_3,DDDD)

In [216]:
# ----------------------------
# Load data
# ----------------------------

d1 = np.load('DNF_1000_voa_Numba.npy', allow_pickle=True).tolist()
d2 = np.load('DF_1000_voa_Numba.npy', allow_pickle=True).tolist()

x1 = d1[0][2][0]
y1 = d1[0][2][1]
x2 = d2[0][2][0]
y2 = d2[0][2][1]

#DB_1 = Vis.convert_to_diff_db(np.load('rx_300m_day_2_long_etas.npy'))
#ts_1 = [i / 1000 for i in range(len(DB_1))]

#DB_2 = Vis.convert_to_diff_db(np.load('rx_300m_day_2_long_etas.npy'))
#ts_2 = [i / 1000 for i in range(len(DB_2))]

DB_1 = -delta_eta_to_loss_db2(del_eta[0])
ts_1 = [i / 10000 for i in range(len(DB_1))]

DB_2 = -delta_eta_to_loss_db2(del_eta[0])
ts_2 = [i / 10000 for i in range(len(DB_2))]

# ----------------------------
# Plot configuration (edit THIS cell only)
# ----------------------------

PLOT_CFG = {
    # Global styling / output
    "figsize": (8, 6),
    "title": "QBER/time - 1000Hz, Day (VOA Ramp Only)",
    "outfile": "C5_FinalDraft_Day_10Hz_NFvsF.pdf",
    "main_xlim": (0, 200),
    "main_ylim": (0, 50),

    # Series labels (top plot)
    "top_labels": ["No Fibre", "Fibre"],

    # Series labels (bottom plot) - if you omit one series, only first label is used
    "bottom_labels": ["No Fibre", "Fibre"],

    # Whether to plot the second (orange) bottom series at all
    "plot_bottom_second_series": False,   # <- set False to omit DB_2/ts_2
    
    # Histogram label
    "hist_xlabel": "Prob. [%]",

    # Legend configuration (place legend on the upper histogram by default)
    # You can change this without touching plotting code.
    "legend": {
        "target": "hist_main",                 # "hist_main", "main", "sub", "hist_sub"
        "loc": "lower center",
        "bbox_to_anchor": (-1.64, 0.83),       #-1.64,-0.015 for lower (manual)
        "fontsize": 12,
        "frameon": True,
        "ncol": 1
    },

    # Layout tweaks
    "gridspec": {
        "height_ratios": [3.5, 1],
        "width_ratios": [6.2, 1.4],
        "hspace": 0.015,
        "wspace": 0.015
    },

    # Histogram styling
    "hist_style": {
        "fill_alpha": 0.55,
        "sigma_band_alpha": 0.18,
        "lw": 0,
        "mean_lw": 2.8
    }
}

PLOT_CFG["hist_bins_top_scale"] = 0.6

HIST_STYLE_CFG = {
    "outline_alpha": 0.80,
    "outline_lw": 2.2,
}

HIST_MARK_CFG = {
    # mean marker position: fraction of xmax from left (close to right edge)
    "x_frac": 0.92,

    # per-series x-offset (orange pushed right)
    "dx_frac": 0.04,

    # keep away from the absolute edge
    "pad_frac": 0.0000,

    # ---- Mean tick ----
    "mean_tick_frac": 0.08,   # <-- longer than before
    "mean_alpha": 1.0,
    "mean_lw": 2.0,

    # ---- Sigma spread ----
    "show_sigma": True,
    "sigma_alpha": 0.55,       # <-- fainter
    "sigma_lw": 1.8
}

# Bundle series inputs so the plotting cell stays generic
SERIES_TOP = [
    {"x": x1, "y": y1 * 100},
    {"x": x2, "y": y2 * 100},
]
SERIES_BOTTOM = [
    {"x": ts_1, "y": DB_1},
    {"x": ts_2, "y": DB_2},
]


In [222]:
# ----------------------------
# Matplotlib style
# ----------------------------
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=18)
plt.rc('axes', labelsize=16)

fig = plt.figure(figsize=PLOT_CFG["figsize"])

gs_cfg = PLOT_CFG["gridspec"]
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=gs_cfg["height_ratios"],
    width_ratios=gs_cfg["width_ratios"],
    hspace=gs_cfg["hspace"],
    wspace=gs_cfg["wspace"]
)

ax_main      = fig.add_subplot(gs[0, 0])
ax_sub       = fig.add_subplot(gs[1, 0], sharex=ax_main)
ax_hist_main = fig.add_subplot(gs[0, 1], sharey=ax_main)
ax_hist_sub  = fig.add_subplot(gs[1, 1], sharey=ax_sub)

# ----------------------------
# Main plots (top)
# ----------------------------
top_labels = PLOT_CFG["top_labels"]
line_top_0, = ax_main.plot(SERIES_TOP[0]["x"], SERIES_TOP[0]["y"], label=top_labels[0])
line_top_1, = ax_main.plot(SERIES_TOP[1]["x"], SERIES_TOP[1]["y"], label=top_labels[1])

ax_main.set_xlim(*PLOT_CFG["main_xlim"])
ax_main.set_ylim(*PLOT_CFG["main_ylim"])
ax_main.tick_params(axis='both', labelsize=14)

QBER_THRESH = 10.0  # percent

ax_main.axhline(
    QBER_THRESH,
    linestyle="--",
    linewidth=2.6,
    color="0.25",
    alpha=0.95,
    zorder=6
)




# Bottom plot (Δη)
bottom_labels = PLOT_CFG["bottom_labels"]
ax_sub.plot(SERIES_BOTTOM[0]["x"], SERIES_BOTTOM[0]["y"],
            color=line_top_0.get_color(), label=bottom_labels[0])

plot_bottom_second = PLOT_CFG["plot_bottom_second_series"]
if plot_bottom_second:
    ax_sub.plot(SERIES_BOTTOM[1]["x"], SERIES_BOTTOM[1]["y"],
                color=line_top_1.get_color(), label=bottom_labels[1])
ax_sub.tick_params(axis='both', labelsize=12)

ax_main.grid(True, axis='y', alpha=0.5, linewidth=0.8)
ax_sub.grid(True, axis='y', alpha=0.35, linewidth=0.8)
# ----------------------------
# Labels / title
# ----------------------------
ax_sub.set_xlabel('Time [$s$]')
ax_main.set_ylabel('QBER [$\\%$]')
ax_sub.set_ylabel('$\\Delta\\eta$ [db]')
ax_main.set_title(PLOT_CFG["title"])

plt.setp(ax_main.get_xticklabels(), visible=False)
ax_main.tick_params(axis='x', which='both', length=0)

# ----------------------------
# Histogram axes formatting
# ----------------------------
for axh in (ax_hist_main, ax_hist_sub):
    axh.grid(False)
    axh.tick_params(axis='y', labelleft=False, left=False)
    axh.tick_params(axis='y', which='both', length=0)
    axh.minorticks_off()
    axh.tick_params(axis='both', labelsize=9)  # de-emphasise histogram numbers
    for spine in axh.spines.values():
        spine.set_linewidth(0.45)

# Avoid tick/label collision between histogram panels
ax_hist_main.tick_params(axis='x', which='both',length=0,labelbottom=False)
ax_hist_main.set_xlabel("")

ax_hist_sub.set_xlabel(PLOT_CFG["hist_xlabel"])
ax_hist_main.grid(True, axis='y', alpha=0.5, linewidth=0.6)
ax_hist_sub.grid(True, axis='y', alpha=0.35, linewidth=0.6)
QBER_THRESH = 10.0  # %
ax_hist_main.axhline(
    QBER_THRESH,
    linestyle="--",
    linewidth=1.4,
    color="0.25",
    alpha=0.25,
    zorder=10
)


# ----------------------------
# Build bins and plot histograms
# ----------------------------
fig.canvas.draw()

ymin_m, ymax_m = ax_main.get_ylim()
ymin_s, ymax_s = ax_sub.get_ylim()

# Select bottom series to include for histogram (mirrors line plotting)
bottom_hist_data = [SERIES_BOTTOM[0]["y"]]
if plot_bottom_second:
    bottom_hist_data.append(SERIES_BOTTOM[1]["y"])

nb_main_base = choose_bins_freedman_diaconis(
    np.r_[SERIES_TOP[0]["y"], SERIES_TOP[1]["y"]], ymin_m, ymax_m
)

nb_main = int(nb_main_base * PLOT_CFG["hist_bins_top_scale"])

nb_sub = choose_bins_freedman_diaconis(
    np.concatenate([_finite(d) for d in bottom_hist_data]), ymin_s, ymax_s
)

bins_main = np.linspace(ymin_m, ymax_m, nb_main)
bins_sub  = np.linspace(ymin_s, ymax_s, nb_sub)

# Colors
c0 = line_top_0.get_color()
c1 = line_top_1.get_color()

# Histogram style: expects keys like fill_alpha/core_alpha/marker_size
# e.g. HIST_STYLE_CFG = {"fill_alpha":0.18, "core_alpha":0.65, "marker_size":16}
p1, mu1, s1 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[0]["y"], bins_main, c0, **HIST_STYLE_CFG)
p2, mu2, s2 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[1]["y"], bins_main, c1, **HIST_STYLE_CFG)

p3, mu3, s3 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[0]["y"], bins_sub, c0, **HIST_STYLE_CFG)
p4, mu4, s4 = (0.0, np.nan, np.nan)

if plot_bottom_second:
    p4, mu4, s4 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[1]["y"], bins_sub, c1, **HIST_STYLE_CFG)


ax_main.set_axisbelow(True)
ax_sub.set_axisbelow(True)
# Shared histogram x-limits
peak = max(p1, p2, p3, p4)
xmax = 1.10 * peak if peak > 0 else 1.0
# Give a little extra room on the right for the offset markers
ax_hist_main.set_xlim(0, xmax * 1.10)
ax_hist_sub.set_xlim(0, xmax * 1.10)

# Draw mean + sigma markers ON the histogram RHS.
# series_index=0 -> first series (blue), series_index=1 -> second (orange) offset to the right.
draw_mean_sigma_on_hist(ax_hist_main, mu1, s1, c0, HIST_MARK_CFG, series_index=0)
draw_mean_sigma_on_hist(ax_hist_main, mu2, s2, c1, HIST_MARK_CFG, series_index=1)

draw_mean_sigma_on_hist(ax_hist_sub,  mu3, s3, c0, HIST_MARK_CFG, series_index=0)
if plot_bottom_second:
    draw_mean_sigma_on_hist(ax_hist_sub, mu4, s4, c1, HIST_MARK_CFG, series_index=1)



# ----------------------------
# Histogram ticks: a few ticks above 0, hide label at 0
# ----------------------------
# Choose ticks in percent, then convert to fraction units
pct_ticks = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]  # adjust if you want
frac_ticks = [p / 100 for p in pct_ticks]

# Keep only ticks within [0, xmax]
frac_ticks = [t for t in frac_ticks if t <= xmax]

for axh in (ax_hist_main, ax_hist_sub):
    axh.set_xticks(frac_ticks)

# Hide the 0 label but keep the tick mark
def _hide_zero_percent(x, pos):
    if np.isclose(x, 0):
        return ""
    return f"{int(round(100 * x))}"

for axh in (ax_hist_main, ax_hist_sub):
    axh.xaxis.set_major_formatter(FuncFormatter(_hide_zero_percent))





# ----------------------------
# Tick padding: separate main vs histogram
# ----------------------------
ax_main.tick_params(axis='x', which='major', pad=4)
ax_sub.tick_params(axis='x', which='major', pad=4)
ax_hist_main.tick_params(axis='x', which='major', pad=0)
ax_hist_sub.tick_params(axis='x', which='major', pad=0)

# ----------------------------
# Legend (configurable target + position) with μ,σ for top series
# ----------------------------
legend_cfg = dict(PLOT_CFG.get("legend", {}))
target = legend_cfg.pop("target", "hist_main")

legend_targets = {
    "main": ax_main,
    "sub": ax_sub,
    "hist_main": ax_hist_main,
    "hist_sub": ax_hist_sub
}
ax_for_legend = legend_targets.get(target, ax_hist_main)

legend_labels = [
    f"{top_labels[0]}: μ={mu1:.2f}%, σ={s1:.2f}%",
    f"{top_labels[1]}: μ={mu2:.2f}%, σ={s2:.2f}%"
]
apply_legend(ax_for_legend, [line_top_0, line_top_1], legend_labels, legend_cfg)


# Margins / save
# ----------------------------
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.10, top=0.92)

plt.show()
#plt.savefig(PLOT_CFG["outfile"], dpi=300)


In [224]:
def time_to_volts(t, t_min=0.0, t_max=200.0, v_min=0.0, v_max=5.0):
    """Linear mapping from Time [s] to Volts [V]."""
    t = np.asarray(t, dtype=float)
    return v_min + (t - t_min) * (v_max - v_min) / (t_max - t_min)

def volts_to_time(v, t_min=0.0, t_max=200.0, v_min=0.0, v_max=5.0):
    """Inverse linear mapping from Volts [V] to Time [s]."""
    v = np.asarray(v, dtype=float)
    return t_min + (v - v_min) * (t_max - t_min) / (v_max - v_min)


In [324]:
# ----------------------------
# Load data
# ----------------------------

d1 = np.load('DNF_1000_voa_Numba.npy', allow_pickle=True).tolist()
d2 = np.load('DF_1000_voa_Numba.npy', allow_pickle=True).tolist()

x1 = d1[0][2][0]
y1 = d1[0][2][1]
x2 = d2[0][2][0]
y2 = d2[0][2][1]

#DB_1 = Vis.convert_to_diff_db(np.load('rx_300m_day_2_long_etas.npy'))
#ts_1 = [i / 1000 for i in range(len(DB_1))]

#DB_2 = Vis.convert_to_diff_db(np.load('rx_300m_day_2_long_etas.npy'))
#ts_2 = [i / 1000 for i in range(len(DB_2))]

DB_1 = delta_eta_to_loss_db2(del_eta[0])
ts_1 = [i / 10000 for i in range(len(DB_1))]

DB_2 = delta_eta_to_loss_db2(del_eta[0])
ts_2 = [i / 10000 for i in range(len(DB_2))]

# ----------------------------
# Plot configuration (edit THIS cell only)
# ----------------------------

PLOT_CFG = {
    # Global styling / output
    "figsize": (8, 6),
    "title": "QBER/Voltage - 1000Hz, Day (VOA Ramp Only)",
    "outfile": "C5_FinalDraft_Day_1000Hz_VOA_NFvF.pdf",
    "main_xlim": (0, 200),
    "main_ylim": (0, 50),

    # Series labels (top plot)
    "top_labels": ["No Fibre", "Fibre"],

    # Series labels (bottom plot) - if you omit one series, only first label is used
    "bottom_labels": ["No Fibre", "Fibre"],

    # Whether to plot the second (orange) bottom series at all
    "plot_bottom_second_series": False,   # <- set False to omit DB_2/ts_2
    
    # Histogram label
    "hist_xlabel": "Prob. [%]",

    # Legend configuration (place legend on the upper histogram by default)
    # You can change this without touching plotting code.
    "legend": {
        "target": "hist_main",                 # "hist_main", "main", "sub", "hist_sub"
        "loc": "lower center",
        "bbox_to_anchor": (-3.8, 0.83),       #-1.64,-0.015 for lower (manual)
        "fontsize": 12,
        "frameon": True,
        "ncol": 1
    },

    # Layout tweaks
    "gridspec": {
        "height_ratios": [3.5, 1],
        "width_ratios": [6.2, 1.4],
        "hspace": 0.015,
        "wspace": 0.015
    },

    # Histogram styling
    "hist_style": {
        "fill_alpha": 0.55,
        "sigma_band_alpha": 0.18,
        "lw": 0,
        "mean_lw": 2.8
    }
}

PLOT_CFG["hist_bins_top_scale"] = 0.6

HIST_STYLE_CFG = {
    "outline_alpha": 0.80,
    "outline_lw": 2.2,
}

HIST_MARK_CFG = {
    # mean marker position: fraction of xmax from left (close to right edge)
    "x_frac": 0.92,

    # per-series x-offset (orange pushed right)
    "dx_frac": 0.04,

    # keep away from the absolute edge
    "pad_frac": 0.0000,

    # ---- Mean tick ----
    "mean_tick_frac": 0.08,   # <-- longer than before
    "mean_alpha": 1.0,
    "mean_lw": 2.0,

    # ---- Sigma spread ----
    "show_sigma": True,
    "sigma_alpha": 0.55,       # <-- fainter
    "sigma_lw": 1.8
}

# Bundle series inputs so the plotting cell stays generic
SERIES_TOP = [
    {"x": x1, "y": y1 * 100},
    {"x": x2, "y": y2 * 100},
]
SERIES_BOTTOM = [
    {"x": ts_1, "y": DB_1},
    {"x": ts_2, "y": DB_2},
]

In [326]:
# ---- Extra axis between panels (Volts) ----
PLOT_CFG["mid_xaxis"] = {
    "enabled": True,
    "label": "VOA Voltage [V]",
    "v_min": 0.0,
    "v_max": 5.0,
    "ticks": [1, 2, 3, 4, 5],   # requested
    # mapping assumes main x-axis is time in seconds
    "t_min": PLOT_CFG["main_xlim"][0],
    "t_max": PLOT_CFG["main_xlim"][1],
}

# ---- Optional log scaling ----
PLOT_CFG["yscale_main"] = "linear"  # "linear" or "log"
PLOT_CFG["yscale_sub"]  = "linear"  # "linear" or "log"

# ---- Histogram x-limits coupling ----
PLOT_CFG["share_hist_x"] = True     # bottom hist uses top hist xlim (may clip)


In [328]:
# ----------------------------
# Matplotlib style
# ----------------------------
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=18)
plt.rc('axes', labelsize=16)

fig = plt.figure(figsize=PLOT_CFG["figsize"])

gs_cfg = PLOT_CFG["gridspec"]
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=gs_cfg["height_ratios"],
    width_ratios=gs_cfg["width_ratios"],
    hspace=gs_cfg["hspace"],
    wspace=gs_cfg["wspace"]
)

ax_main      = fig.add_subplot(gs[0, 0])
ax_sub       = fig.add_subplot(gs[1, 0], sharex=ax_main)
ax_hist_main = fig.add_subplot(gs[0, 1], sharey=ax_main)
ax_hist_sub  = fig.add_subplot(gs[1, 1], sharey=ax_sub)

# --- Convert time axis to voltage ---
T_MAX = PLOT_CFG["main_xlim"][1]  # 200 s
V_MAX = 5.0                       # 5 V full ramp

def time_to_voltage(t):
    return V_MAX * (np.asarray(t) / T_MAX)

# ----------------------------
# Main plots (top)
# ----------------------------
top_labels = PLOT_CFG["top_labels"]
xv0 = time_to_voltage(SERIES_TOP[0]["x"])
xv1 = time_to_voltage(SERIES_TOP[1]["x"])

line_top_0, = ax_main.plot(xv0, SERIES_TOP[0]["y"], label=top_labels[0])
line_top_1, = ax_main.plot(xv1, SERIES_TOP[1]["y"], label=top_labels[1])


ax_main.set_xlim(0, 5)
ax_main.set_ylim(*PLOT_CFG["main_ylim"])
ax_main.tick_params(axis='both', labelsize=14)

# Optional y scaling
ax_main.set_yscale(PLOT_CFG.get("yscale_main", "linear"))

QBER_THRESH = 10.0  # percent
VOA_THRESH = 2.60  # your existing mapping

ax_main.axhline(QBER_THRESH, linestyle="--", linewidth=2.6, color="0.25", alpha=0.95, zorder=6)
ax_main.axvline(VOA_THRESH,  linestyle="--", linewidth=2.2, color="0.25", alpha=0.45, zorder=6)

# ----------------------------
# Bottom plot (Δη)
# ----------------------------
ax_sub.plot(time_to_voltage(SERIES_BOTTOM[0]["x"]), SERIES_BOTTOM[0]["y"],
            color=line_top_0.get_color(), label=bottom_labels[0])

if plot_bottom_second:
    ax_sub.plot(time_to_voltage(SERIES_BOTTOM[1]["x"]), SERIES_BOTTOM[1]["y"],
                color=line_top_1.get_color(), label=bottom_labels[1])


ax_sub.tick_params(axis='both', labelsize=12)
ax_sub.minorticks_on()

# Optional y scaling
ax_sub.set_yscale(PLOT_CFG.get("yscale_sub", "linear"))

# Grid (horizontal only)
ax_main.grid(True, axis='y', alpha=0.5, linewidth=0.8)
ax_sub.grid(True, axis='y', alpha=0.35, linewidth=0.8)

# ----------------------------
# Labels / title
# ----------------------------
ax_sub.set_xlabel('VOA Voltage [V]')
ax_main.set_ylabel('QBER [$\\%$]')
ax_sub.set_ylabel('$\\Delta\\eta$ [db]')
ax_main.set_title(PLOT_CFG["title"])

# Hide top x tick labels (shared x)
plt.setp(ax_main.get_xticklabels(), visible=False)
ax_main.tick_params(axis='x', which='both', length=0)

# ----------------------------
# Histogram axes formatting
# ----------------------------
for axh in (ax_hist_main, ax_hist_sub):
    axh.grid(False)
    axh.tick_params(axis='y', labelleft=False, left=False)
    axh.tick_params(axis='y', which='both', length=0)
    axh.minorticks_off()
    axh.tick_params(axis='both', labelsize=9)
    for spine in axh.spines.values():
        spine.set_linewidth(0.45)

# Avoid tick/label collision between histogram panels
ax_hist_main.tick_params(axis='x', which='both', length=0, labelbottom=False)
ax_hist_main.set_xlabel("")
ax_hist_sub.set_xlabel(PLOT_CFG["hist_xlabel"])

# Optional horizontal grid on histogram y (kept subtle)
ax_hist_main.grid(True, axis='y', alpha=0.5, linewidth=0.6)
ax_hist_sub.grid(True, axis='y', alpha=0.35, linewidth=0.6)

# Threshold line on histogram
ax_hist_main.axhline(QBER_THRESH, linestyle="--", linewidth=1.4, color="0.25", alpha=0.25, zorder=10)

# Mark VOA threshold on bottom plot
ax_sub.axvline(VOA_THRESH, linestyle="--", linewidth=1.4, color="0.25", alpha=0.25, zorder=10)

# ----------------------------
# Build bins and plot histograms
# ----------------------------
fig.canvas.draw()

ymin_m, ymax_m = ax_main.get_ylim()
ymin_s, ymax_s = ax_sub.get_ylim()

bottom_hist_data = [SERIES_BOTTOM[0]["y"]]
if plot_bottom_second:
    bottom_hist_data.append(SERIES_BOTTOM[1]["y"])

nb_main_base = choose_bins_freedman_diaconis(
    np.r_[SERIES_TOP[0]["y"], SERIES_TOP[1]["y"]], ymin_m, ymax_m
)
nb_main = int(nb_main_base * PLOT_CFG["hist_bins_top_scale"])

nb_sub = choose_bins_freedman_diaconis(
    np.concatenate([_finite(d) for d in bottom_hist_data]), ymin_s, ymax_s
)

bins_main = np.linspace(ymin_m, ymax_m, nb_main)
bins_sub  = np.linspace(ymin_s, ymax_s, nb_sub)

c0 = line_top_0.get_color()
c1 = line_top_1.get_color()

p1, mu1, s1 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[0]["y"], bins_main, c0, **HIST_STYLE_CFG)
p2, mu2, s2 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[1]["y"], bins_main, c1, **HIST_STYLE_CFG)

p3, mu3, s3 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[0]["y"], bins_sub, c0, **HIST_STYLE_CFG)
p4, mu4, s4 = (0.0, np.nan, np.nan)
if plot_bottom_second:
    p4, mu4, s4 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[1]["y"], bins_sub, c1, **HIST_STYLE_CFG)

# Shared histogram x-limits
peak_top = max(p1, p2)
peak_all = max(p1, p2, p3, p4)

# Top histogram defines the scale
xmax_top = 1.10 * peak_top if peak_top > 0 else 1.0

# If requested: bottom histogram uses top's scale (may clip)
if PLOT_CFG.get("share_hist_x", True):
    ax_hist_main.set_xlim(0, xmax_top * 1.10)
    ax_hist_sub.set_xlim(0, xmax_top * 1.10)
else:
    xmax_all = 1.10 * peak_all if peak_all > 0 else 1.0
    ax_hist_main.set_xlim(0, xmax_all * 1.10)
    ax_hist_sub.set_xlim(0, xmax_all * 1.10)

# Draw mean+sigma markers
draw_mean_sigma_on_hist(ax_hist_main, mu1, s1, c0, HIST_MARK_CFG, series_index=0)
draw_mean_sigma_on_hist(ax_hist_main, mu2, s2, c1, HIST_MARK_CFG, series_index=1)

draw_mean_sigma_on_hist(ax_hist_sub,  mu3, s3, c0, HIST_MARK_CFG, series_index=0)
if plot_bottom_second:
    draw_mean_sigma_on_hist(ax_hist_sub, mu4, s4, c1, HIST_MARK_CFG, series_index=1)

# ----------------------------
# Histogram ticks: percent scale
# ----------------------------
xmax_used = ax_hist_main.get_xlim()[1]  # now the controlling x-range
pct_ticks = [0, 5, 10, 15, 20, 25, 30]
frac_ticks = [p / 100 for p in pct_ticks]
frac_ticks = [t for t in frac_ticks if t <= xmax_used]

for axh in (ax_hist_main, ax_hist_sub):
    axh.set_xticks(frac_ticks)

def _hide_zero_percent(x, pos):
    if np.isclose(x, 0):
        return ""
    return f"{int(round(100 * x))}"

for axh in (ax_hist_main, ax_hist_sub):
    axh.xaxis.set_major_formatter(FuncFormatter(_hide_zero_percent))

# ----------------------------
# Legend with μ,σ for top series
# ----------------------------
legend_cfg = dict(PLOT_CFG.get("legend", {}))
target = legend_cfg.pop("target", "hist_main")

legend_targets = {"main": ax_main, "sub": ax_sub, "hist_main": ax_hist_main, "hist_sub": ax_hist_sub}
ax_for_legend = legend_targets.get(target, ax_hist_main)

legend_labels = [
    f"{top_labels[0]}",
    f"{top_labels[1]}"
]
apply_legend(ax_for_legend, [line_top_0, line_top_1], legend_labels, legend_cfg)

# ----------------------------
# Margins / show
# ----------------------------
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.10, top=0.92)

plt.show()
plt.savefig(PLOT_CFG["outfile"], dpi=300)


In [11]:
import numpy as np
import matplotlib.pyplot as plt

def plot_visibility_vs_tau_with_poisson_regions(
    res,
    *,
    title="Visibility vs Tau (Poisson-dominated regions highlighted)",
    N_crit: float = 50.0,          # counts/window below which shot noise is inherently large
    poisson_frac: float = 0.6,     # if >=60% of variance explained by Poisson -> call it Poisson-dominated
    shade_alpha: float = 0.18,
    show_sigma_panel: bool = True,
):
    """
    Takes VisibilityTauResult `res` (from lockin_visibility_summary_vs_tau_mle_numba)
    and highlights tau regions where the large 10–90% band is likely dominated by Poisson noise.

    Poisson test is based on:
      - expected counts per window N = res.expected_counts_per_window
      - empirical sigma from (V_p90 - V_p10)/2.563
      - predicted Poisson sigma ~ sqrt((1 - V^2)/N)

    A tau is flagged Poisson-dominated if:
      (N < N_crit) OR (sigma_pois^2 / sigma_emp^2 >= poisson_frac)
    """

    taus = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)

    # finite mask
    m = np.isfinite(taus) & np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) & np.isfinite(Nexp) & (taus > 0)
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus = taus[m]
    Vmean = Vmean[m]
    Vmed  = Vmed[m]
    Vmax  = Vmax[m]
    Vmin  = Vmin[m]
    Vp10  = Vp10[m]
    Vp90  = Vp90[m]
    Nexp  = Nexp[m]

    # ---- Empirical sigma from 10–90 band
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007  # 2*(z_0.90)=2*1.28155...

    # ---- Poisson-predicted sigma (rule-of-thumb)
    # Clamp V into [0,1] for the formula, because it is a physical variance model
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    # ---- How much of variance could be explained by Poisson?
    var_emp = np.maximum(sigma_emp*sigma_emp, 1e-20)
    var_pois = sigma_pois*sigma_pois
    frac_explained = np.clip(var_pois / var_emp, 0.0, 1.0)

    poisson_dom = (Nexp < N_crit) | (frac_explained >= poisson_frac)

    # helper: shade contiguous regions where poisson_dom True
    def _shade_regions(ax, x, mask, *, alpha=0.18):
        # x is sorted in log axis order already (taus increasing)
        idx = np.where(mask)[0]
        if idx.size == 0:
            return
        # find contiguous runs
        runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
        for r in runs:
            a = x[r[0]]
            b = x[r[-1]]
            # extend to next point edge for nicer shading
            ax.axvspan(a, b, alpha=alpha)

    # ---- Plot: either one panel or two panels
    if show_sigma_panel:
        fig, ax = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
        fig.set_size_inches(8, 6)
    else:
        fig, ax = plt.subplots(1, 1)
        fig.set_size_inches(8, 6)
        ax = [ax]

    # ===== Top: visibility summary =====
    ax0 = ax[0]
    ax0.plot(taus, Vmean, label="Vmean", linewidth=1.6)
    ax0.plot(taus, Vmed,  label="Vmedian", linewidth=1.2)
    ax0.plot(taus, Vmax,  label="Vmax", linewidth=1.0)
    ax0.plot(taus, Vmin,  label="Vmin", linewidth=1.0)
    ax0.fill_between(taus, Vp10, Vp90, alpha=0.20, label="10–90% band")

    _shade_regions(ax0, taus, poisson_dom, alpha=shade_alpha)

    ax0.set_xscale("log")
    ax0.set_ylim(-0.05, 1.25)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, alpha=0.25)
    ax0.legend(loc="best")

    # remove excess x padding
    ax0.set_xlim(taus[0], taus[-1])

    # ===== Bottom: noise diagnostics =====
    if show_sigma_panel:
        ax1 = ax[1]
        ax1.plot(taus, sigma_emp, label=r"$\sigma_{\mathrm{emp}}$ from 10–90 band", linewidth=1.6)
        ax1.plot(taus, sigma_pois, label=r"$\sigma_{\mathrm{Pois}} \approx \sqrt{(1-V^2)/N}$", linewidth=1.3)
        ax1.plot(taus, frac_explained, label=r"Poisson variance fraction", linewidth=1.2)

        _shade_regions(ax1, taus, poisson_dom, alpha=shade_alpha)

        ax1.set_xscale("log")
        ax1.set_yscale("log")
        ax1.set_xlabel("Tau (s)")
        ax1.set_ylabel("σ / fraction")
        ax1.grid(True, alpha=0.25)
        ax1.legend(loc="best")

    fig.set_constrained_layout(True)

    # Print a quick summary
    print("[Poisson-region diagnostics]")
    print(f"  N_crit = {N_crit} counts/window")
    print(f"  poisson_frac threshold = {poisson_frac:.2f}")
    print(f"  Fraction of taus flagged Poisson-dominated: {np.mean(poisson_dom):.2%}")
    print(f"  Smallest tau flagged: {taus[poisson_dom][0] if np.any(poisson_dom) else np.nan}")
    print(f"  Largest tau flagged:  {taus[poisson_dom][-1] if np.any(poisson_dom) else np.nan}")

    plt.show()

    # return arrays in case you want to annotate/report
    return {
        "taus": taus,
        "sigma_emp": sigma_emp,
        "sigma_pois": sigma_pois,
        "poisson_frac_explained": frac_explained,
        "poisson_dominated_mask": poisson_dom,
        "Nexp": Nexp,
    }


In [13]:
diag = plot_visibility_vs_tau_with_poisson_regions(
    d1[1],
    N_crit=50,           # tune for your regime
    poisson_frac=0.6,    # tune for strictness
    show_sigma_panel=True
)

[Poisson-region diagnostics]
  N_crit = 50 counts/window
  poisson_frac threshold = 0.60
  Fraction of taus flagged Poisson-dominated: 20.41%
  Smallest tau flagged: 0.00030000000000000003
  Largest tau flagged:  0.003


In [15]:
import numpy as np
import matplotlib.pyplot as plt

def plot_visibility_vs_tau_with_poisson_and_mle_error(
    res,
    *,
    title="Visibility vs Tau (Poisson-dominated regions + MLE error estimate)",
    N_crit: float = 150.0,         # good default for your regime (see notes)
    poisson_frac: float = 0.6,
    shade_alpha: float = 0.18,
    mle_k: float = 1.0,            # scale factor for the MLE sigma model (keep 1.0 unless calibrating)
    show_sigma_panel: bool = True,
):
    """
    Plots V(tau) summaries and highlights Poisson-dominated regions.
    Adds an estimated MLE-only sigma_V(tau) ~ sqrt((1 - V^2)/N).

    Uses:
      - res.V_p10/res.V_p90 for empirical spread
      - res.expected_counts_per_window for expected photons per window
    """

    taus = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)

    m = np.isfinite(taus) & np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) & np.isfinite(Nexp) & (taus > 0)
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus = taus[m]
    Vmean = Vmean[m]
    Vmed  = Vmed[m]
    Vmax  = Vmax[m]
    Vmin  = Vmin[m]
    Vp10  = Vp10[m]
    Vp90  = Vp90[m]
    Nexp  = Nexp[m]

    # ---- empirical sigma from 10–90 band
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007  # 2*(z_0.90)=2*1.28155...

    # ---- Poisson-pred sigma (rule-of-thumb)
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    # ---- MLE sigma estimate (same scaling, optionally with a tunable factor)
    # In practice, MLE and Poisson scaling are very similar; this just makes it explicit as "MLE error".
    sigma_mle = mle_k * sigma_pois

    # ---- fraction of empirical variance explained by Poisson/MLE
    var_emp = np.maximum(sigma_emp*sigma_emp, 1e-20)
    frac_explained = np.clip((sigma_pois*sigma_pois) / var_emp, 0.0, 1.0)

    poisson_dom = (Nexp < N_crit) | (frac_explained >= poisson_frac)

    def _shade_regions(ax, x, mask, *, alpha=0.18):
        idx = np.where(mask)[0]
        if idx.size == 0:
            return
        runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
        for r in runs:
            ax.axvspan(x[r[0]], x[r[-1]], alpha=alpha)

    # ---- figure layout
    if show_sigma_panel:
        fig, ax = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
        fig.set_size_inches(8, 6)
        ax0, ax1 = ax
    else:
        fig, ax0 = plt.subplots(1, 1)
        fig.set_size_inches(8, 6)
        ax1 = None

    # ===== Top: visibility summaries =====
    ax0.plot(taus, Vmean, label="Vmean", linewidth=2.0)
    ax0.plot(taus, Vmed,  label="Vmedian", linewidth=1.5)
    ax0.plot(taus, Vmax,  label="Vmax", linewidth=1.3)
    ax0.plot(taus, Vmin,  label="Vmin", linewidth=1.3)
    ax0.fill_between(taus, Vp10, Vp90, alpha=0.20, label="10–90% band")

    # ---- MLE error band around Vmean (±1σ)
    V_lo = Vmean - sigma_mle
    V_hi = Vmean + sigma_mle
    ax0.fill_between(taus, V_lo, V_hi, alpha=0.15, label="Estimated MLE ±1σ")

    # ---- shade Poisson-dominated regions
    _shade_regions(ax0, taus, poisson_dom, alpha=shade_alpha)

    ax0.set_xscale("log")
    ax0.set_ylim(0, 1.01)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, alpha=0.25)
    ax0.legend(loc="best")
    ax0.set_xlim(taus[0], taus[-1])

    # ===== Bottom: sigma diagnostics =====
    if ax1 is not None:
        ax1.plot(taus, sigma_emp, label=r"$\sigma_{\mathrm{emp}}$ (from 10–90 band)", linewidth=2.6)
        ax1.plot(taus, sigma_pois, label=r"$\sigma_{\mathrm{Pois}}$", linewidth=2.3)
        ax1.plot(taus, sigma_mle, label=r"$\sigma_{\mathrm{MLE, est}}$", linewidth=2.2, linestyle="--")
        ax1.plot(taus, frac_explained, label="Poisson variance fraction", linewidth=2.1)

        _shade_regions(ax1, taus, poisson_dom, alpha=shade_alpha)

        ax1.set_xscale("log")
        ax1.set_yscale("log")
        ax1.set_xlabel("Tau (s)")
        ax1.set_ylabel("σ / fraction")
        ax1.grid(True, alpha=0.25)
        ax1.legend(loc="best")

    fig.set_constrained_layout(True)
    plt.show()

    print("[Noise diagnostics]")
    print(f"  Mean rate implied by expected_counts/tau ~ {np.nanmedian(Nexp/taus):.2e} counts/s")
    print(f"  N_crit = {N_crit:.1f} counts/window")
    print(f"  poisson_frac threshold = {poisson_frac:.2f}")
    print(f"  Fraction of taus flagged Poisson-dominated: {np.mean(poisson_dom):.2%}")

    return {
        "taus": taus,
        "sigma_emp": sigma_emp,
        "sigma_pois": sigma_pois,
        "sigma_mle": sigma_mle,
        "poisson_frac_explained": frac_explained,
        "poisson_dominated_mask": poisson_dom,
        "Nexp": Nexp,
    }


In [17]:
diag = plot_visibility_vs_tau_with_poisson_and_mle_error(
    d2[1],
    N_crit=150,
    poisson_frac=0.8,
    mle_k=1.0,
    show_sigma_panel=True
)

[Noise diagnostics]
  Mean rate implied by expected_counts/tau ~ 2.73e+04 counts/s
  N_crit = 150.0 counts/window
  poisson_frac threshold = 0.80
  Fraction of taus flagged Poisson-dominated: 24.49%


In [19]:
d1[1]

VisibilityTauResult(taus=array([3.00e-04, 4.00e-04, 5.00e-04, 6.00e-04, 7.00e-04, 8.00e-04,
       9.00e-04, 1.00e-03, 2.00e-03, 3.00e-03, 4.00e-03, 5.00e-03,
       6.00e-03, 7.00e-03, 8.00e-03, 1.00e-02, 1.10e-02, 1.30e-02,
       1.50e-02, 1.70e-02, 1.90e-02, 2.20e-02, 2.60e-02, 2.90e-02,
       3.40e-02, 3.90e-02, 4.50e-02, 5.20e-02, 6.00e-02, 6.90e-02,
       7.90e-02, 9.10e-02, 1.05e-01, 1.21e-01, 1.39e-01, 1.60e-01,
       1.84e-01, 2.12e-01, 2.44e-01, 2.81e-01, 3.24e-01, 3.73e-01,
       4.29e-01, 4.94e-01, 5.69e-01, 6.55e-01, 7.54e-01, 8.69e-01,
       1.00e+00]), n_windows=array([1999998,  500000,  400000,  333333,  285714,  250000,  222222,
        200000,  100000,   66666,   50000,   40000,   33333,   28571,
         25000,   20000,   18181,   15384,   13333,   11764,   10526,
          9090,    7692,    6896,    5882,    5128,    4444,    3846,
          3333,    2898,    2531,    2197,    1904,    1652,    1438,
          1250,    1086,     943,     819,     711,     617,

In [21]:
import numpy as np
import matplotlib.pyplot as plt

def plot_visibility_vs_tau_poisson_markers(
    res,
    *,
    title: str = "Visibility vs $\\tau$",
    N_crit: float = 50.0,
    poisson_frac: float = 0.6,
    figsize=(8, 6),
    # styling to match thesis plots
    top_tick_size: int = 14,
    bottom_tick_size: int = 13,
    grid_alpha_top: float = 0.60,
    grid_alpha_bottom: float = 0.40,
    grid_lw: float = 0.8,
    vline_alpha: float = 0.65,
    vline_lw: float = 1.4,
    vline_ls: str = "--",
    # separate palettes so legends don’t “blend”
    top_colors=("C0", "C1", "C2", "C3"),     # μ, median, Vmax, Vmin
    band_color="C0",                         # 10–90% band color family
    bottom_colors=("C0", "C8"),              # σ_emp, σ_Pois
    # margins like your reference script
    margins=(0.12, 0.95, 0.10, 0.92),        # left, right, bottom, top
    return_diag: bool = False,
    legend_loc_main="upper left",
    legend_anchor_main=(0.02, 0.98),
    
    legend_loc_sigma="upper left",
    legend_anchor_sigma=(0.02, 0.95),
):
    """
    Thesis-style 2×1 plot:
      top: visibility vs tau with 10–90% band
      bottom: sigma_emp and sigma_Pois vs tau (log-y)

    Poisson-dominated taus are marked with vertical dashed lines (not shading).

    A tau is flagged Poisson-dominated if:
      (Nexp < N_crit) OR (sigma_pois^2 / sigma_emp^2 >= poisson_frac)
    """

    # ----------------------------
    # Matplotlib style contract
    # ----------------------------
    plt.rcParams.update({'font.size': 10})
    plt.rc('axes', titlesize=18)
    plt.rc('axes', labelsize=16)
    plt.rcParams["legend.fontsize"] = 12

    # ----------------------------
    # Extract and clean
    # ----------------------------
    taus = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)

    m = (
        np.isfinite(taus) & (taus > 0) &
        np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) &
        np.isfinite(Nexp)
    )
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus  = taus[m]
    Vmean = Vmean[m]
    Vmed  = Vmed[m]
    Vmax  = Vmax[m]
    Vmin  = Vmin[m]
    Vp10  = Vp10[m]
    Vp90  = Vp90[m]
    Nexp  = Nexp[m]

    # enforce increasing taus just in case
    order = np.argsort(taus)
    taus, Vmean, Vmed, Vmax, Vmin, Vp10, Vp90, Nexp = \
        taus[order], Vmean[order], Vmed[order], Vmax[order], Vmin[order], Vp10[order], Vp90[order], Nexp[order]

    # ----------------------------
    # Empirical sigma from 10–90%
    # ----------------------------
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007  # 2*(z_0.90)=2*1.28155...

    # ----------------------------
    # Poisson sigma model (rule-of-thumb)
    # ----------------------------
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    var_emp  = np.maximum(sigma_emp*sigma_emp, 1e-20)
    var_pois = sigma_pois*sigma_pois
    frac_explained = np.clip(var_pois / var_emp, 0.0, 1.0)

    poisson_dom = (Nexp < N_crit) | (frac_explained >= poisson_frac)

    # ----------------------------
    # Find boundary taus to mark with vertical lines
    # Mark transitions in poisson_dom mask (and also single-point segments)
    # ----------------------------
    idx = np.where(poisson_dom)[0]
    vline_taus = []
    if idx.size:
        runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
        for r in runs:
            vline_taus.append(float(taus[r[0]]))
            vline_taus.append(float(taus[r[-1]]))

    # de-duplicate and keep within range
    vline_taus = np.array(sorted(set(vline_taus)), dtype=float)

    # ----------------------------
    # Figure / GridSpec (2×1)
    # ----------------------------
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 1], hspace=0.02)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)

    # ----------------------------
    # Top plot: visibility vs tau
    # ----------------------------
    c_mu, c_med, c_vmax, c_vmin = top_colors

    ax0.plot(taus, Vmean, label=r"$V_{\mathrm{mean}}$", linewidth=2.2, color=c_mu)
    ax0.plot(taus, Vmed,  label=r"$V_{\mathrm{med}}$", linewidth=2.0, color=c_med, linestyle="-.")
    ax0.plot(taus, Vmax,  label=r"$V_{\max}$", linewidth=2.4, color=c_vmax)
    ax0.plot(taus, Vmin,  label=r"$V_{\min}$", linewidth=2.4, color=c_vmin)

    # 10–90% band (keep but visually light)
    ax0.fill_between(taus, Vp10, Vp90, alpha=0.2, label=r"$[V_{10},\,V_{90}]$", color=band_color)

    # vertical dashed lines for Poisson-dominated region boundaries
    for x in vline_taus:
        ax0.axvline(x, linestyle=vline_ls, linewidth=vline_lw, color="0.25", alpha=vline_alpha, zorder=10)

    ax0.set_xscale("log")
    ax0.set_ylim(-0.005, 1.05)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, axis='both', alpha=grid_alpha_top, linewidth=grid_lw)
    ax0.tick_params(axis='both', labelsize=top_tick_size)

    # remove excess x padding
    ax0.set_xlim(taus[0], taus[-1])

    # hide top x tick labels (like your reference)
    plt.setp(ax0.get_xticklabels(), visible=False)
    ax0.tick_params(axis='x', which='both', length=0)

    ax0.legend(
        loc=legend_loc_main,
        bbox_to_anchor=legend_anchor_main,
        frameon=True,
    )

    # ----------------------------
    # Bottom plot: sigma diagnostics
    # ----------------------------
    c_emp, c_pois = bottom_colors

    ax1.plot(taus, sigma_emp,  label=r"$\sigma_{\mathrm{emp}}$", linewidth=2.2, color=c_emp,alpha=0.55)
    ax1.plot(taus, sigma_pois, label=r"$\sigma_{\mathrm{Pois}}$", linewidth=2.2, color=c_pois)

    for x in vline_taus:
        ax1.axvline(x, linestyle=vline_ls, linewidth=vline_lw, color="0.25", alpha=vline_alpha, zorder=10)

    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_ylim(0.005, 9)
    ax1.set_xlabel(r"$\tau$ (s)")
    ax1.set_ylabel(r"$\sigma$")
    ax1.grid(True, axis='both', alpha=grid_alpha_bottom, linewidth=grid_lw)
    ax1.tick_params(axis='both', labelsize=bottom_tick_size)

    ax1.legend(
        loc=legend_loc_sigma,
        bbox_to_anchor=legend_anchor_sigma,
        frameon=True,
    )

    # ----------------------------
    # Manual margins (match your thesis plotting approach)
    # ----------------------------
    left, right, bottom, top = margins
    fig.subplots_adjust(left=left, right=right, bottom=bottom, top=top)

    # ----------------------------
    # Console diagnostics
    # ----------------------------
    print("[Poisson-region diagnostics]")
    print(f"  N_crit = {N_crit} counts/window")
    print(f"  poisson_frac threshold = {poisson_frac:.2f}")
    print(f"  Fraction of taus flagged Poisson-dominated: {np.mean(poisson_dom):.2%}")
    if np.any(poisson_dom):
        print(f"  Smallest tau flagged: {taus[poisson_dom][0]:.3e} s")
        print(f"  Largest  tau flagged: {taus[poisson_dom][-1]:.3e} s")

    plt.show()

    if return_diag:
        return {
            "taus": taus,
            "sigma_emp": sigma_emp,
            "sigma_pois": sigma_pois,
            "poisson_frac_explained": frac_explained,
            "poisson_dominated_mask": poisson_dom,
            "Nexp": Nexp,
            "vline_taus": vline_taus,
        }


In [37]:
diag = plot_visibility_vs_tau_poisson_markers(
    d2[1],
    title="Effective visibility vs integration time $\\tau$",
    N_crit=100,            # expected counts/window threshold
    poisson_frac=1,    # fraction of variance explained by Poisson
    figsize=(8, 6),      # thesis figure size
    return_diag=True,
    legend_loc_main="upper left",
    legend_anchor_main=(0.03, 0.48),
    
    legend_loc_sigma="upper left",
    legend_anchor_sigma=(0.835, 1.045)
)


TypeError: plot_visibility_vs_tau_poisson_markers() got an unexpected keyword argument 'legend_loc_sigma'

In [25]:
d3 = np.load('NNF_10_em_Numba.npy', allow_pickle=True).tolist()


In [27]:
diag = plot_visibility_vs_tau_poisson_markers(
    d3[1],
    title="Effective visibility vs integration time $\\tau$",
    N_crit=100,            # expected counts/window threshold
    poisson_frac=1,    # fraction of variance explained by Poisson
    figsize=(8, 6),      # thesis figure size
    return_diag=True,
    legend_loc_main="upper left",
    legend_anchor_main=(0.03, 0.48),
    
    legend_loc_sigma="upper left",
    legend_anchor_sigma=(0.835, 1.045)
)


[Poisson-region diagnostics]
  N_crit = 100 counts/window
  poisson_frac threshold = 1.00
  Fraction of taus flagged Poisson-dominated: 20.41%
  Smallest tau flagged: 3.000e-04 s
  Largest  tau flagged: 3.000e-03 s


In [29]:
import numpy as np
import matplotlib.pyplot as plt

def plot_visibility_vs_tau_poisson_markers(
    res,
    *,
    f_ramp_hz: float,
    title: str = "Visibility vs $\\tau$",
    N_crit: float = 50.0,
    poisson_frac: float = 0.6,
    figsize=(8, 6),
    # styling to match thesis plots
    top_tick_size: int = 14,
    bottom_tick_size: int = 13,
    grid_alpha_top: float = 0.60,
    grid_alpha_bottom: float = 0.40,
    grid_lw: float = 0.8,
    vline_alpha: float = 0.65,
    vline_lw: float = 1.4,
    vline_ls: str = "--",
    # separate palettes so legends don’t “blend”
    top_colors=("C0", "C1", "C2", "C3"),     # mean, median, Vmax, Vmin
    band_color="C0",                         # 10–90% band color family
    bottom_color="C8",                       # ratio line color
    # shading (BOTTOM ONLY)
    shade_alpha: float = 0.12,
    # margins like your reference script
    margins=(0.12, 0.95, 0.10, 0.92),        # left, right, bottom, top
    return_diag: bool = False,
    legend_loc_main="upper left",
    legend_anchor_main=(0.03, 0.48),
    legend_loc_bottom="upper left",
    legend_anchor_bottom=(0.835, 1.045),
    # how to choose the single Poisson boundary tau (vertical line)
    boundary_mode: str = "first_non_poisson",   # "first_non_poisson" or "sigma_cross"
):
    """
    Thesis-style 2×1 plot:
      top: visibility vs tau with 10–90% band
      bottom: ratio r = sigma_pois^2 / sigma_emp^2 (single line, log-y)

    Two vertical dashed lines are drawn on BOTH axes:
      1) tau_pois_boundary: boundary for Poisson-dominated regime
      2) tau_ramp = 1/f_ramp_hz: one ramp cycle

    Shading appears ONLY on the bottom plot:
      - [taus_min, tau_pois_boundary]  (Poisson-dominated region)
      - [taus_min, tau_ramp]          (sub-cycle region)
    The top plot is *not* shaded; only the dashed lines indicate the regimes there.

    Poisson-dominated test:
      (Nexp < N_crit) OR (sigma_pois^2 / sigma_emp^2 >= poisson_frac)

    boundary_mode:
      - "first_non_poisson": tau_pois_boundary is the first tau where poisson_dom becomes False
      - "sigma_cross": tau_pois_boundary is where sigma_pois^2 / sigma_emp^2 crosses 1 (closest)
    """

    if f_ramp_hz <= 0:
        raise ValueError("f_ramp_hz must be > 0.")

    # ----------------------------
    # Matplotlib style contract
    # ----------------------------
    plt.rcParams.update({'font.size': 10})
    plt.rc('axes', titlesize=18)
    plt.rc('axes', labelsize=16)
    plt.rcParams["legend.fontsize"] = 12

    # ----------------------------
    # Extract and clean
    # ----------------------------
    taus = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)

    m = (
        np.isfinite(taus) & (taus > 0) &
        np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) &
        np.isfinite(Nexp)
    )
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus  = taus[m]
    Vmean = Vmean[m]
    Vmed  = Vmed[m]
    Vmax  = Vmax[m]
    Vmin  = Vmin[m]
    Vp10  = Vp10[m]
    Vp90  = Vp90[m]
    Nexp  = Nexp[m]

    # enforce increasing taus just in case
    order = np.argsort(taus)
    taus, Vmean, Vmed, Vmax, Vmin, Vp10, Vp90, Nexp = \
        taus[order], Vmean[order], Vmed[order], Vmax[order], Vmin[order], Vp10[order], Vp90[order], Nexp[order]

    # ----------------------------
    # Empirical sigma from 10–90%
    # ----------------------------
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007  # 2*(z_0.90)=2*1.28155...

    # ----------------------------
    # Poisson sigma model (rule-of-thumb)
    # ----------------------------
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    # ratio: how much of variance could be explained by Poisson
    var_emp  = np.maximum(sigma_emp*sigma_emp, 1e-20)
    var_pois = sigma_pois*sigma_pois
    ratio = np.clip(var_pois / var_emp, 1e-6, 1e6)  # keep plottable, interpret as "Poisson variance fraction"

    poisson_dom = (Nexp < N_crit) | (ratio >= poisson_frac)

    # ----------------------------
    # Determine the two key taus (vertical lines)
    # ----------------------------
    tau_ramp = 1.0 / float(f_ramp_hz)

    # Poisson boundary tau (single representative boundary for plotting)
    tau_pois = np.nan
    if boundary_mode == "first_non_poisson":
        # first tau that is NOT poisson-dominated; boundary is at that tau
        idx_non = np.where(~poisson_dom)[0]
        if idx_non.size:
            tau_pois = float(taus[idx_non[0]])
        else:
            tau_pois = float(taus[-1])  # everything is flagged -> boundary at end
    elif boundary_mode == "sigma_cross":
        # find tau where ratio is closest to 1 (sigma_pois^2 == sigma_emp^2)
        j = int(np.argmin(np.abs(np.log(ratio))))  # log-scale closeness to 1
        tau_pois = float(taus[j])
    else:
        raise ValueError("boundary_mode must be 'first_non_poisson' or 'sigma_cross'.")

    # Clamp lines to plotting range (still draw if within)
    tau_min_plot = float(taus[0])
    tau_max_plot = float(taus[-1])

    def _in_range(x):
        return (x >= tau_min_plot) and (x <= tau_max_plot)

    # ----------------------------
    # Figure / GridSpec (2×1)
    # ----------------------------
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 1], hspace=0.02)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)

    # ----------------------------
    # Top plot: visibility vs tau
    # ----------------------------
    c_mean, c_med, c_vmax, c_vmin = top_colors

    ax0.plot(taus, Vmean, label=r"$\bar{V}$", linewidth=2.2, color=c_mean)
    ax0.plot(taus, Vmed,  label=r"$\tilde{V}$", linewidth=2.0, color=c_med, linestyle="-.")
    ax0.plot(taus, Vmax,  label=r"$V_{\max}$", linewidth=2.4, color=c_vmax)
    ax0.plot(taus, Vmin,  label=r"$V_{\min}$", linewidth=2.4, color=c_vmin)

    ax0.fill_between(taus, Vp10, Vp90, alpha=0.2, label=r"$[V_{10},\,V_{90}]$", color=band_color)

    # dashed lines (no shading on top)
    if _in_range(tau_pois):
        ax0.axvline(tau_pois, linestyle=vline_ls, linewidth=vline_lw, color="0.25", alpha=vline_alpha, zorder=10)
    if _in_range(tau_ramp):
        ax0.axvline(tau_ramp, linestyle=vline_ls, linewidth=vline_lw, color="0.25", alpha=vline_alpha, zorder=10)

    ax0.set_xscale("log")
    ax0.set_ylim(-0.005, 1.05)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, axis='both', alpha=grid_alpha_top, linewidth=grid_lw)
    ax0.tick_params(axis='both', labelsize=top_tick_size)

    ax0.set_xlim(tau_min_plot, tau_max_plot)

    plt.setp(ax0.get_xticklabels(), visible=False)
    ax0.tick_params(axis='x', which='both', length=0)

    ax0.legend(
        loc=legend_loc_main,
        bbox_to_anchor=legend_anchor_main,
        frameon=True,
    )

    # ----------------------------
    # Bottom plot: ratio diagnostics (single line)
    # ----------------------------
    ax1.plot(taus, ratio, label="_nolegend_", linewidth=2.2, color=bottom_color)

    # shaded regions on bottom ONLY
    # 1) Poisson-dominated region: [min_tau, tau_pois]
    if _in_range(tau_pois):
        ax1.axvspan(tau_min_plot, tau_pois, color="0.5", alpha=shade_alpha, zorder=0)
    # 2) sub-cycle region: [min_tau, tau_ramp]
    if _in_range(tau_ramp):
        ax1.axvspan(tau_min_plot, tau_ramp, color="0.5", alpha=shade_alpha, zorder=0)

    # dashed lines on bottom too
    if _in_range(tau_pois):
        ax1.axvline(tau_pois, linestyle=vline_ls, linewidth=vline_lw, color="0.25", alpha=vline_alpha, zorder=10)
    if _in_range(tau_ramp):
        ax1.axvline(tau_ramp, linestyle=vline_ls, linewidth=vline_lw, color="0.25", alpha=vline_alpha, zorder=10)

    ax1.set_xscale("log")
    ax1.set_yscale("log")
    #ax1.set_ylim(1e-4, 10.0)  # ratio is in [0,1] for your clipped version, but keep room
    ax1.set_xlabel(r"$\tau$ (s)")
    ax1.set_ylabel(r"$\sigma_{\mathrm{Pois}}^2/\sigma_{\mathrm{emp}}^2$")
    ax1.grid(True, axis='both', alpha=grid_alpha_bottom, linewidth=grid_lw)
    ax1.tick_params(axis='both', labelsize=bottom_tick_size)

    # legend text must include what this tau is (as you requested)
    # We'll define it explicitly in the legend:
    #   tau_pois: "Poisson boundary tau" (as chosen by boundary_mode)
    #   tau_ramp: "1/f_ramp" (one ramp cycle)
    legend_lines = []
    legend_labels = []

    # Add the ratio line as legend handle
    h_ratio = ax1.lines[-1]
    legend_lines.append(h_ratio)
    legend_labels.append(r"$\sigma_{\mathrm{Pois}}^2/\sigma_{\mathrm{emp}}^2$")

    # Add tau markers as proxy handles
    from matplotlib.lines import Line2D
    proxy = Line2D([0], [0], color="0.25", linestyle=vline_ls, linewidth=vline_lw, alpha=vline_alpha)
    legend_lines.append(proxy)
    legend_labels.append(
        rf"$\tau_{{\mathrm{{Pois}}}} \approx {tau_pois:.2e}\,\mathrm{{s}}$  ({boundary_mode})"
    )

    proxy2 = Line2D([0], [0], color="0.25", linestyle=vline_ls, linewidth=vline_lw, alpha=vline_alpha)
    legend_lines.append(proxy2)
    legend_labels.append(
        rf"$\tau_{{\mathrm{{ramp}}}} = 1/f_{{\mathrm{{ramp}}}} \approx {tau_ramp:.2e}\,\mathrm{{s}}$"
    )

    ax1.legend(
        legend_lines, legend_labels,
        loc=legend_loc_bottom,
        bbox_to_anchor=legend_anchor_bottom,
        frameon=True,
    )

    # ----------------------------
    # Manual margins (match your thesis plotting approach)
    # ----------------------------
    left, right, bottom, top = margins
    fig.subplots_adjust(left=left, right=right, bottom=bottom, top=top)

    # ----------------------------
    # Console diagnostics
    # ----------------------------
    print("[Poisson / ramp diagnostics]")
    print(f"  f_ramp = {f_ramp_hz:.6g} Hz  ->  tau_ramp = {tau_ramp:.6e} s")
    print(f"  N_crit = {N_crit} counts/window")
    print(f"  poisson_frac threshold = {poisson_frac:.2f}  (ratio >= threshold => Poisson-dominated)")
    print(f"  boundary_mode = {boundary_mode}  ->  tau_pois ≈ {tau_pois:.6e} s")
    print(f"  Fraction of taus flagged Poisson-dominated: {np.mean(poisson_dom):.2%}")
    if np.any(poisson_dom):
        print(f"  Smallest tau flagged: {taus[poisson_dom][0]:.3e} s")
        print(f"  Largest  tau flagged: {taus[poisson_dom][-1]:.3e} s")

    plt.show()

    if return_diag:
        return {
            "taus": taus,
            "Vmean": Vmean,
            "Vmed": Vmed,
            "Vmax": Vmax,
            "Vmin": Vmin,
            "Vp10": Vp10,
            "Vp90": Vp90,
            "Nexp": Nexp,
            "sigma_emp": sigma_emp,
            "sigma_pois": sigma_pois,
            "ratio_var_pois_over_emp": ratio,
            "poisson_dominated_mask": poisson_dom,
            "tau_pois": tau_pois,
            "tau_ramp": tau_ramp,
            "boundary_mode": boundary_mode,
        }


In [38]:
diag = plot_visibility_vs_tau_poisson_markers(
    d3[1],
    f_ramp_hz=10,
    title="Effective visibility vs integration time $\\tau$",
    N_crit=10,            # expected counts/window threshold
    poisson_frac=1,    # fraction of variance explained by Poisson
    figsize=(8, 6),      # thesis figure size
    return_diag=True,
)


[Poisson / ramp diagnostics]
  f_ramp = 10 Hz  ->  tau_ramp = 1.000000e-01 s
  N_crit = 10 counts/window
  poisson_frac threshold = 1.00  (ratio >= threshold => Poisson-dominated)
  boundary_mode = first_non_poisson  ->  tau_pois ≈ 5.000000e-04 s
  Fraction of taus flagged Poisson-dominated: 16.33%
  Smallest tau flagged: 3.000e-04 s
  Largest  tau flagged: 2.000e-03 s


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

def log_decimal_formatter(val, pos=None):
    """Log ticks as decimals, with 1 shown as 1.0"""
    if val <= 0:
        return ""

    exp = np.log10(val)
    if not np.isclose(exp, round(exp)):  # only powers of 10
        return ""
    exp = int(round(exp))

    # Special case: show 1 as 1.0
    if exp == 0:
        return "1.0"

    if exp > 0:
        return str(int(10**exp))   # 10, 100, ...
    else:
        # decimals like 0.1, 0.01
        return f"{10**exp:.{abs(exp)}f}".rstrip("0").rstrip(".")

def plot_visibility_vs_tau_poisson_and_ramp_markers(
    res,
    *,
    f_ramp: float,
    title: str = r"Effective visibility vs integration time $\tau$",
    figsize=(8, 6),

    # styling to match thesis plots
    top_tick_size: int = 14,
    bottom_tick_size: int = 11,
    grid_alpha_top: float = 0.450,
    grid_alpha_bottom: float = 0.30,
    grid_lw: float = 0.8,

    # figure margins: left, right, bottom, top
    margins=(0.12, 0.95, 0.10, 0.92),

    # top plot colors
    top_colors=("C0", "C1", "C2", "C3"),
    band_color="C0",

    # bottom curve color
    ratio_color="C8",
    ratio_color2="C8",

    # vertical line styling (":" for both; different colors)
    tau_pois_color="C7",
    tau_ramp_color="C9",
    vline_lw=1.8,
    vline_alpha=0.70,

    # lower-axis shading (only)
    shade_alpha: float = 0.075,
    shade_pois_color="0.2",
    shade_ramp_color="0.5",

    # legend placement
    legend_loc_main="upper left",
    legend_anchor_main=(0.03, 0.48),
    legend_loc_bottom="upper left",
    legend_anchor_bottom=(0.835, 1.045),
    legend_spacing=0.3,
    legend_box=0.2,

    # y-limits
    ylim_top=(-0.05, 1.05),
    ylim_ratio=(1e-2, 1e2),

    # ratio label: keep terse (caption explains)
    ratio_label: str = r"$\sigma_{r}(\tau)$",
    ratio_label2: str = r"$V_{r}(\tau)^{2}$",

    # choose what ratio you want
    use_variance_ratio: bool = False,  # True: σ²_Pois/σ²_emp ; False: σ_Pois/σ_emp

    return_diag: bool = False,
):
    """
    Top: visibility summary vs tau with 10–90% band.
    Bottom: a single diagnostic curve σ_r(τ):
        - if use_variance_ratio: σ_r(τ) = σ_Pois² / σ_emp²
        - else:                 σ_r(τ) = σ_Pois  / σ_emp

    τ_Pois is chosen as the *last* crossing of σ_r(τ)=1 (log–log interpolation).
    τ_ramp = 1/f_ramp.

    Only the lower plot is lightly shaded (two regions); upper plot uses lines only.
    """

    if f_ramp <= 0:
        raise ValueError("f_ramp must be > 0.")
    tau_ramp = 1.0 / float(f_ramp)

    # Style contract
    plt.rcParams.update({'font.size': 10})
    plt.rc('axes', titlesize=18)
    plt.rc('axes', labelsize=16)
    plt.rcParams["legend.fontsize"] = 10.5

    # Extract
    taus  = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)
    rVav = Vmean/Vmed

    m = (
        np.isfinite(taus) & (taus > 0) &
        np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) &
        np.isfinite(Nexp)
    )
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus, Vmean, Vmed, Vmax, Vmin, Vp10, Vp90, Nexp = \
        taus[m], Vmean[m], Vmed[m], Vmax[m], Vmin[m], Vp10[m], Vp90[m], Nexp[m]

    order = np.argsort(taus)
    taus, Vmean, Vmed, Vmax, Vmin, Vp10, Vp90, Nexp = \
        taus[order], Vmean[order], Vmed[order], Vmax[order], Vmin[order], Vp10[order], Vp90[order], Nexp[order]

    # Empirical sigma from 10–90%
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007

    # Poisson sigma model
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    # Ratio (variance or amplitude)
    if use_variance_ratio:
        num = sigma_pois * sigma_pois
        den = np.maximum(sigma_emp * sigma_emp, 1e-24)
    else:
        num = sigma_pois
        den = np.maximum(sigma_emp, 1e-12)

    r = num / den

    # finite for crossing
    mr = np.isfinite(r) & (r > 0)
    taus_r = taus[mr]
    r_r    = r[mr]
    if taus_r.size < 2:
        raise ValueError("Not enough finite ratio points to find crossings.")

    # ---- last crossing of r(tau)=1 using log–log interpolation
    logt = np.log10(taus_r)
    logr = np.log10(r_r)
    s = np.sign(logr)
    s[s == 0] = 1.0
    flips = np.where(s[:-1] * s[1:] < 0)[0]

    tau_pois = np.nan
    if flips.size:
        j = flips[-1]
        x0, x1 = logt[j], logt[j+1]
        y0, y1 = logr[j], logr[j+1]
        if np.isfinite(y0) and np.isfinite(y1) and (y1 != y0):
            x_star = x0 + (0.0 - y0) * (x1 - x0) / (y1 - y0)
            tau_pois = 10**x_star

    # ---- Figure
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 1], hspace=0.05)
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)

    # ---- Top
    c_mu, c_med, c_vmax, c_vmin = top_colors
    ax0.plot(taus, Vmean, label=r"$V_{\mathrm{mean}}$", linewidth=2.2, color=c_mu)
    ax0.plot(taus, Vmed,  label=r"$V_{\mathrm{med}}$",  linewidth=2.0, color=c_med, linestyle="-")
    ax0.plot(taus, Vmax,  label=r"$V_{\max}$",          linewidth=2.4, color=c_vmax)
    ax0.plot(taus, Vmin,  label=r"$V_{\min}$",          linewidth=2.4, color=c_vmin)
    ax0.fill_between(taus, Vp10, Vp90, alpha=0.20, label=r"$V_{{\!10–90}}$", color=band_color)

    # vertical markers (no shading on top)
    if np.isfinite(tau_pois):
        ax0.axvline(tau_pois, linestyle="--", linewidth=vline_lw, color=tau_pois_color, alpha=vline_alpha, zorder=10)
    ax0.axvline(tau_ramp, linestyle="--", linewidth=vline_lw, color=tau_ramp_color, alpha=vline_alpha, zorder=10)

    ax0.set_xscale("log")
    ax0.set_ylim(*ylim_top)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, axis='both', alpha=grid_alpha_top, linewidth=grid_lw)
    ax0.tick_params(axis='both', labelsize=top_tick_size)
    ax0.set_xlim(taus[0], taus[-1])

    plt.setp(ax0.get_xticklabels(), visible=False)
    ax0.tick_params(axis='x', which='both', length=0)

    ax0.legend(loc=legend_loc_main, bbox_to_anchor=legend_anchor_main, frameon=True, labelspacing=legend_spacing,borderpad=legend_box,borderaxespad=0.15)

    # ---- Bottom (two curves)
    ax1.plot(taus_r, rVav**2, color=ratio_color2, linewidth=2.2, label=ratio_label2, zorder=5)
    ax1.plot(taus_r, r_r, color=ratio_color, linewidth=2.2, label=ratio_label, zorder=5)


    # Shade only bottom:
    # - below-ramp region
    ax1.axvspan(taus[0], min(tau_ramp, taus[-1]), alpha=shade_alpha, color=shade_ramp_color, zorder=0)
    # - Poisson-side region (typically tau < tau_pois)
    if np.isfinite(tau_pois):
        ax1.axvspan(taus[0], min(tau_pois, taus[-1]), alpha=shade_alpha, color=shade_pois_color, zorder=0)

    # vertical lines on bottom
    if np.isfinite(tau_pois):
        ax1.axvline(tau_pois, linestyle="--", linewidth=vline_lw, color=tau_pois_color, alpha=vline_alpha, zorder=10)
    ax1.axvline(tau_ramp, linestyle="--", linewidth=vline_lw, color=tau_ramp_color, alpha=vline_alpha, zorder=10)

    # reference line r=1 (helps reading; keep subtle)
    ax1.axhline(1.0, color="0.25", linewidth=1.6, alpha=0.85, zorder=11,linestyle="--")

    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_ylim(*ylim_ratio)
    ax1.set_xlabel(r"$\tau$ (s)")
    ax1.set_ylabel(r"$r(\tau)$")
    ax1.grid(True, axis='both', alpha=grid_alpha_bottom, linewidth=grid_lw)
    ax1.tick_params(axis='both', labelsize=bottom_tick_size)
    ax1.yaxis.set_major_formatter(FuncFormatter(log_decimal_formatter))

    # Legend ONLY for ratio symbol (no taus)
    ax1.legend(loc=legend_loc_bottom, bbox_to_anchor=legend_anchor_bottom, frameon=True,labelspacing=legend_spacing,borderpad=legend_box,borderaxespad=0.15)

    # margins
    left, right, bottom, top = margins
    fig.subplots_adjust(left=left, right=right, bottom=bottom, top=top)

    print("[Markers]")
    print(f"  tau_ramp = 1/f_ramp: {tau_ramp:.6e} s")
    if np.isfinite(tau_pois):
        print(f"  tau_pois (last crossing r=1): {tau_pois:.6e} s")
    else:
        print("  tau_pois: no r=1 crossing found")

    plt.show()

    if return_diag:
        return {
            "taus": taus,
            "r": r,
            "taus_r": taus_r,
            "r_r": r_r,
            "tau_pois": tau_pois,
            "tau_ramp": tau_ramp,
            "use_variance_ratio": use_variance_ratio,
        }


In [860]:
d3 = np.load('NL_10_em_Numba.npy', allow_pickle=True).tolist()
%matplotlib qt
diag = plot_visibility_vs_tau_poisson_and_ramp_markers(
    d3[1],
    f_ramp=10,
    title="Vis./$\\tau$ - 10Hz, Night Slow (emulation, fibre)",
    figsize=(8, 6),      # thesis figure size
    return_diag=True,
    legend_anchor_main=(0.8595, 1.0025),
    legend_anchor_bottom=(0.8575, 0.995),
    #legend_anchor_bottom=(0.8075, 0.425),
    tau_pois_color="Purple",
    tau_ramp_color="olive",
    vline_lw=1.8,
    vline_alpha=0.75,
    ratio_color="C4",
    ratio_color2="olivedrab",
    shade_alpha = 0.125,
    shade_pois_color="0.3",
    shade_ramp_color="0.6",
    ylim_ratio=(3e-2, 3e1),
    ylim_top=(-0.02, 1.15),
    legend_box=0.25,
    legend_spacing=0.1,


)
plt.savefig('C5_NightSlow_10Hz_em_VisTau.pdf', dpi=300)

[Markers]
  tau_ramp = 1/f_ramp: 1.000000e-01 s
  tau_pois (last crossing r=1): 1.858398e-03 s


In [1]:
#left
#legend_anchor_main=(0.01, 0.5),
#legend_anchor_bottom=(0.835, 1.055)
#right
    legend_anchor_main=(0.8595, 0.415),
    legend_anchor_bottom=(0.8575, 0.555),

    legend_anchor_main=(0.8595, 1.0025),
    legend_anchor_bottom=(0.8575, 0.65),

IndentationError: unexpected indent (3281069580.py, line 5)

In [5]:
# Auxiliary cell: Figure 6.3 — control-referenced QBER summary
# Requires NumPy, pandas and Matplotlib; no changes to existing plotting cells.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

DATA_DIR = Path(".")  # Folder containing your processed .npy files

# One entry per emulation/control pair.
# Use the TRUE no-emulation control, not the VOA-sweep measurement.
#
# Example entry — replace the control filename before uncommenting:
#
# RUNS = [
#     dict(
#         case="Day",
#         frequency=1000,
#         fibre=False,
#         tau_s=0.003,
#         emulation="DNF_1000_em_Numba.npy",
#         control="REPLACE_WITH_MATCHING_CONTROL.npy",
#     ),
# ]
#
# You can also supply already-loaded objects instead of filenames:
# emulation=loaded_emulation, control=loaded_control

RUNS = []

CASE_ORDER = [
    "Night", "Medium", "Day",
    "Night Slow", "Night Fast", "Night Very Fast",
]

FREQUENCIES = (10, 1000)
EXPECTED_TAU = {10: 0.100, 1000: 0.003}

# Restrict BOTH recordings to this elapsed-time interval.
# None uses every finite sample, matching your histogram statistics.
TIME_RANGE = None  # Alternatively: (0.0, 200.0)

# Your notebook multiplies the saved QBER values by 100.
QBER_INPUT_UNITS = "fraction"  # "fraction" or "percent"

SAVE_FIGURE = False
OUTPUT_STEM = Path("Figure_6_3_QBER_summary")


def extract_qber(source, *, data_dir=DATA_DIR,
                 units=QBER_INPUT_UNITS, time_range=TIME_RANGE):
    """Extract the same time/QBER arrays used by the existing plots."""
    if isinstance(source, (str, Path)):
        path = Path(source)
        if not path.is_absolute():
            path = data_dir / path
        # Only load trusted files: object arrays may contain pickled objects.
        data = np.load(path, allow_pickle=True).tolist()
    else:
        data = source

    try:
        time = np.asarray(data[0][2][0], dtype=float)
        qber = np.asarray(data[0][2][1], dtype=float)
    except (IndexError, KeyError, TypeError, ValueError) as exc:
        raise ValueError(
            "Expected notebook structure: "
            "data[0][2][0] = time; data[0][2][1] = QBER."
        ) from exc

    if time.ndim != 1 or qber.ndim != 1 or time.shape != qber.shape:
        raise ValueError("Time and QBER must be equal-length 1-D arrays.")

    keep = np.isfinite(time) & np.isfinite(qber)
    if time_range is not None:
        start, stop = time_range
        if start >= stop:
            raise ValueError("TIME_RANGE must satisfy start < stop.")
        keep &= (time >= start) & (time <= stop)

    qber = qber[keep]
    if qber.size < 2:
        raise ValueError("Fewer than two valid QBER samples remain.")

    if units == "fraction":
        qber = 100.0 * qber
    elif units != "percent":
        raise ValueError("units must be 'fraction' or 'percent'.")

    if np.any((qber < 0) | (qber > 100)):
        raise ValueError(
            "QBER outside 0–100%. Check input units and saved data."
        )

    return qber


def build_qber_summary(runs):
    """Compute separate recording statistics, then their differences."""
    rows = []
    seen = set()

    for run in runs:
        case = run["case"]
        frequency = int(run["frequency"])
        fibre = run["fibre"]
        tau = float(run["tau_s"])

        if case not in CASE_ORDER:
            raise ValueError(f"Unknown case: {case!r}")
        if frequency not in FREQUENCIES:
            raise ValueError(f"Unsupported frequency: {frequency}")
        if not isinstance(fibre, (bool, np.bool_)):
            raise ValueError("'fibre' must be True or False.")
        if not np.isclose(tau, EXPECTED_TAU[frequency]):
            raise ValueError(
                f"{case}, {frequency} Hz: expected tau="
                f"{EXPECTED_TAU[frequency]} s; supplied {tau} s."
            )

        key = (case, frequency, bool(fibre))
        if key in seen:
            raise ValueError(
                f"Duplicate condition {key}. Repeated runs need an "
                "explicit aggregation/uncertainty method."
            )
        seen.add(key)

        em = extract_qber(run["emulation"])
        ctrl = extract_qber(run["control"])

        # ddof=0 matches np.std(data) in your existing histogram function.
        mu_em, sd_em = float(em.mean()), float(em.std(ddof=0))
        mu_ctrl, sd_ctrl = float(ctrl.mean()), float(ctrl.std(ddof=0))

        rows.append({
            "case": case,
            "frequency_Hz": frequency,
            "fibre": bool(fibre),
            "tau_s": tau,
            "n_emulation": em.size,
            "n_control": ctrl.size,
            "mean_emulation_pct": mu_em,
            "mean_control_pct": mu_ctrl,
            "std_emulation_pp": sd_em,
            "std_control_pp": sd_ctrl,
            "delta_mean_pp": mu_em - mu_ctrl,
            "delta_std_pp": sd_em - sd_ctrl,
        })

    return pd.DataFrame(rows)


def plot_qber_summary(summary):
    """Two columns: reference frequency; two rows: mean and spread."""
    if summary.empty:
        raise ValueError("No measurements supplied.")

    # Retain only cases actually supplied, in the desired order.
    cases = [
        case for case in CASE_ORDER
        if case in set(summary["case"])
    ]
    positions = {case: i for i, case in enumerate(cases)}

    styles = {
        False: dict(color="C0", marker="o", face="C0", label="No fibre"),
        True: dict(color="C1", marker="s", face="none", label="1 km fibre"),
    }

    with plt.rc_context({
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }):
        fig, axes = plt.subplots(
            2, 2, figsize=(11, 7),
            sharex=True, sharey="row",
        )

        metrics = ("delta_mean_pp", "delta_std_pp")

        for col, frequency in enumerate(FREQUENCIES):
            subset = summary[summary["frequency_Hz"] == frequency]

            axes[0, col].set_title(
                rf"$f_{{\mathrm{{ref}}}}={frequency}\,\mathrm{{Hz}}$"
                "\n"
                rf"$\tau={1000*EXPECTED_TAU[frequency]:g}\,\mathrm{{ms}}$"
            )

            for row, metric in enumerate(metrics):
                ax = axes[row, col]
                ax.axhline(0, color="0.4", ls="--", lw=1)
                ax.grid(axis="y", alpha=0.25)
                ax.set_axisbelow(True)

                for fibre, style in styles.items():
                    selected = subset[subset["fibre"] == fibre]
                    offset = 0.10 if fibre else -0.10
                    x = [
                        positions[case] + offset
                        for case in selected["case"]
                    ]
                    ax.scatter(
                        x, selected[metric],
                        marker=style["marker"],
                        s=55,
                        facecolors=style["face"],
                        edgecolors=style["color"],
                        linewidths=1.5,
                        zorder=3,
                    )

                if subset.empty:
                    ax.text(
                        0.5, 0.5, "No data supplied",
                        transform=ax.transAxes, ha="center",
                    )

                # Separate the two experiment groups where both exist.
                strength = [
                    positions[c] for c in cases
                    if c in ("Night", "Medium", "Day")
                ]
                wind = [
                    positions[c] for c in cases
                    if c.startswith("Night ")
                ]
                if strength and wind:
                    ax.axvline(
                        (max(strength) + min(wind)) / 2,
                        color="0.8", lw=1,
                    )

                ax.set_xlim(-0.5, len(cases) - 0.5)

            labels = [
                c.replace("Night ", "Night\n") for c in cases
            ]
            axes[1, col].set_xticks(range(len(cases)), labels)
            axes[1, col].set_xlabel("Emulated channel case")

        axes[0, 0].set_ylabel(
            r"$\Delta\mu_Q$" + "\n[percentage points]"
        )
        axes[1, 0].set_ylabel(
            r"$\Delta\sigma_Q$" + "\n[percentage points]"
        )

        handles = [
            Line2D(
                [], [], linestyle="none",
                marker=s["marker"], markersize=7,
                markerfacecolor=s["face"],
                markeredgecolor=s["color"],
                markeredgewidth=1.5, label=s["label"],
            )
            for s in styles.values()
        ]
        fig.legend(
            handles=handles, loc="upper center",
            ncol=2, frameon=False,
        )
        fig.tight_layout(rect=(0, 0, 1, 0.93))

    return fig, axes


# ------------------------------------------------------------------
# Run after populating RUNS
# ------------------------------------------------------------------

if not RUNS:
    print(
        "Populate RUNS with matching emulation/no-emulation pairs, "
        "then rerun this cell."
    )
else:
    qber_summary = build_qber_summary(RUNS)
    fig63, axes63 = plot_qber_summary(qber_summary)

    print(qber_summary.to_string(index=False, float_format="%.4g"))

    if SAVE_FIGURE:
        fig63.savefig(
            OUTPUT_STEM.with_suffix(".pdf"), bbox_inches="tight"
        )
        fig63.savefig(
            OUTPUT_STEM.with_suffix(".png"),
            dpi=300, bbox_inches="tight",
        )
        qber_summary.to_csv(
            OUTPUT_STEM.with_suffix(".csv"), index=False
        )

    plt.show()

Populate RUNS with matching emulation/no-emulation pairs, then rerun this cell.
